# Projeto Integrador
## Análise do atendimento à carga nos horários de pico e vale do SIN em 2025

### Pergunta de análise

> **Como varia a composição da geração para atendimento à carga entre os horários de pico e de vale no Sistema Interligado Nacional ao longo de 2025?**

### Objetivo

Analisar e comparar a composição da geração hidráulica, térmica, eólica e solar nos horários de maior e menor carga diária do Sistema Interligado Nacional (SIN) em 2025, buscando identificar como as diferentes fontes contribuem para o atendimento da carga nessas condições e como esse comportamento varia ao longo do ano.

### Definição de pico e vale

Para cada dia $d$, o **pico de carga** será definido como o maior valor horário de carga observado:

$$
C_{\text{pico},d} = \max_h \left(C_{d,h}\right)
$$

O **vale de carga** será definido como o menor valor horário observado:

$$
C_{\text{vale},d} = \min_h \left(C_{d,h}\right)
$$

onde:

- $C_{d,h}$ representa a carga no dia $d$ e hora $h$;
- $C_{\text{pico},d}$ representa a carga máxima do dia $d$;
- $C_{\text{vale},d}$ representa a carga mínima do dia $d$.

### Escopo

A análise principal considera o **SIN no ano de 2025** e utiliza a **carga bruta** para identificação dos horários de pico e vale.

Como análise complementar, poderá ser considerada a carga líquida, definida por:

$$
C_{\text{líquida}}
=
C_{\text{bruta}}
-
G_{\text{eólica}}
-
G_{\text{solar}}
$$

Essa análise permitirá avaliar se os horários de maior demanda total coincidem com os horários de maior demanda a ser atendida pelas demais fontes de geração.

## 1. Importação das bibliotecas

Nesta etapa são importadas as bibliotecas que serão utilizadas para manipulação, análise e visualização dos dados.

In [43]:
# Manipulação e análise de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Manipulação de caminhos de arquivos
from pathlib import Path

## 2. Definição dos caminhos do projeto

Os caminhos dos arquivos são definidos de forma relativa à estrutura do projeto.

Dessa forma, o código não depende do local em que o projeto está armazenado no computador.

In [44]:
# Diretório raiz do projeto.
PASTA_PROJETO = Path.cwd().parent

# Diretórios utilizados no projeto
PASTA_DADOS_BRUTOS = PASTA_PROJETO / "dados" / "brutos"
PASTA_DADOS_PROCESSADOS = PASTA_PROJETO / "dados" / "processados"
PASTA_SAIDAS = PASTA_PROJETO / "saidas"

# Exibe os caminhos para conferência
print(f"Projeto:            {PASTA_PROJETO}")
print(f"Dados brutos:       {PASTA_DADOS_BRUTOS}")
print(f"Dados processados:  {PASTA_DADOS_PROCESSADOS}")
print(f"Saídas:             {PASTA_SAIDAS}")

Projeto:            c:\OneDrive\OneDrive - Operador Nacional do Sistema Eletrico\Área de Trabalho\Curso Analise de Dados\projeto-integrador
Dados brutos:       c:\OneDrive\OneDrive - Operador Nacional do Sistema Eletrico\Área de Trabalho\Curso Analise de Dados\projeto-integrador\dados\brutos
Dados processados:  c:\OneDrive\OneDrive - Operador Nacional do Sistema Eletrico\Área de Trabalho\Curso Analise de Dados\projeto-integrador\dados\processados
Saídas:             c:\OneDrive\OneDrive - Operador Nacional do Sistema Eletrico\Área de Trabalho\Curso Analise de Dados\projeto-integrador\saidas


## 3. Verificação dos arquivos disponíveis

Antes de realizar a leitura dos dados, são verificados os arquivos existentes na pasta de dados brutos.

Essa etapa permite confirmar os nomes e extensões dos arquivos utilizados na análise.

In [45]:
# Lista todos os arquivos existentes na pasta de dados brutos
arquivos_brutos = sorted(PASTA_DADOS_BRUTOS.iterdir())

print("Arquivos encontrados:\n")

for arquivo in arquivos_brutos:
    print(f"- {arquivo.name}")

Arquivos encontrados:

- BALANCO_ENERGIA_SUBSISTEMA_2023.csv
- BALANCO_ENERGIA_SUBSISTEMA_2024.csv
- BALANCO_ENERGIA_SUBSISTEMA_2025.csv
- DicionarioDados_Balanco_Energia_Subsistema.json
- DicionarioDados_Balanco_Energia_Subsistema.pdf


## 4. Carregamento da base de dados de 2025

A análise principal do projeto considera os dados do ano de 2025.

Nesta etapa, a base original é carregada diretamente da pasta `dados/brutos`. 
Nenhuma alteração é realizada no arquivo de origem.

Inicialmente, será realizada apenas a leitura e a inspeção da estrutura dos dados. 
Os tratamentos eventualmente necessários serão definidos somente após o diagnóstico 
de qualidade da base e, quando aplicável, os dados tratados serão armazenados em 
`dados/processados`.

In [46]:
# Define o caminho para a base de dados de 2025
ARQUIVO_2025 = PASTA_DADOS_BRUTOS / "BALANCO_ENERGIA_SUBSISTEMA_2025.csv"

# Verifica se o arquivo existe antes de realizar a leitura
if not ARQUIVO_2025.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {ARQUIVO_2025}")

print(f"Arquivo selecionado: {ARQUIVO_2025.name}")

Arquivo selecionado: BALANCO_ENERGIA_SUBSISTEMA_2025.csv


### 4.1 Leitura dos dados

A base de 2025 é carregada em um `DataFrame` do pandas.

### Observação sobre o formato do arquivo

Durante a leitura inicial, verificou-se que o arquivo CSV utiliza ponto e vírgula (`;`) como delimitador dos campos.

Por esse motivo, o parâmetro `sep=";"` foi especificado explicitamente na função `pd.read_csv()`, garantindo que cada variável fosse carregada em uma coluna distinta.

In [47]:
# Realiza a leitura da base de dados de 2025.
# O arquivo utiliza ponto e vírgula (;) como separador entre as colunas.
df_2025 = pd.read_csv(
    ARQUIVO_2025,
    sep=";"
)

# Exibe as cinco primeiras observações para uma inspeção inicial
df_2025.head()

,id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
0,NE,NORDESTE,2025-01-01 00:00:00,2219.280,415.154,16070.653,0.0,12251.182,6453.905
1,N,NORTE,2025-01-01 00:00:00,6818.532,1576.087,401.685,0.0,7097.751,1698.553
2,SIN,SISTEMA INTERLIGADO NACIONAL,2025-01-01 00:00:00,40778.712,6835.338,17662.519,1.0,65277.570,0.000
3,SE,SUDESTE/CENTRO-OESTE,2025-01-01 00:00:00,23801.970,4510.835,5.001,0.0,35374.033,-7056.227
4,S,SUL,2025-01-01 00:00:00,7938.930,333.262,1185.180,1.0,10554.603,-1096.231


## 5. Caracterização inicial da base

Antes de realizar qualquer tratamento, é necessário compreender a estrutura e o conteúdo 
da base de dados.

Nesta etapa serão avaliados:

- número de linhas e colunas;
- nomes das variáveis;
- tipos de dados;
- primeiras e últimas observações;
- período coberto pela base;
- categorias existentes para os subsistemas.

Esse diagnóstico inicial permite verificar se a estrutura observada está de acordo com 
o esperado antes do início das transformações.

In [48]:
# Obtém o número de linhas e colunas da base
n_linhas, n_colunas = df_2025.shape

print(f"Número de linhas:  {n_linhas:,}")
print(f"Número de colunas: {n_colunas}")

Número de linhas:  43,800
Número de colunas: 9


In [49]:
# Lista as variáveis existentes na base
print("Variáveis disponíveis:\n")

for coluna in df_2025.columns:
    print(f"- {coluna}")

Variáveis disponíveis:

- id_subsistema
- nom_subsistema
- din_instante
- val_gerhidraulica
- val_gertermica
- val_gereolica
- val_gersolar
- val_carga
- val_intercambio


In [50]:
# Exibe informações sobre a estrutura do DataFrame,
# incluindo tipos de dados e quantidade de valores não nulos
df_2025.info()

<class 'pandas.DataFrame'>
RangeIndex: 43800 entries, 0 to 43799
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_subsistema      43800 non-null  str    
 1   nom_subsistema     43800 non-null  str    
 2   din_instante       43800 non-null  str    
 3   val_gerhidraulica  43800 non-null  float64
 4   val_gertermica     43800 non-null  float64
 5   val_gereolica      43800 non-null  float64
 6   val_gersolar       43800 non-null  float64
 7   val_carga          43800 non-null  float64
 8   val_intercambio    43800 non-null  float64
dtypes: float64(6), str(3)
memory usage: 3.0 MB


In [51]:
# Exibe as primeiras observações da base
df_2025.head()

,id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
0,NE,NORDESTE,2025-01-01 00:00:00,2219.280,415.154,16070.653,0.0,12251.182,6453.905
1,N,NORTE,2025-01-01 00:00:00,6818.532,1576.087,401.685,0.0,7097.751,1698.553
2,SIN,SISTEMA INTERLIGADO NACIONAL,2025-01-01 00:00:00,40778.712,6835.338,17662.519,1.0,65277.570,0.000
3,SE,SUDESTE/CENTRO-OESTE,2025-01-01 00:00:00,23801.970,4510.835,5.001,0.0,35374.033,-7056.227
4,S,SUL,2025-01-01 00:00:00,7938.930,333.262,1185.180,1.0,10554.603,-1096.231


In [52]:
# Exibe as últimas observações da base
df_2025.tail()

,id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
43795,NE,NORDESTE,2025-12-31 23:00:00,3492.122,568.987,10798.684,0.0,13066.976,1792.817
43796,N,NORTE,2025-12-31 23:00:00,11435.862,1860.173,299.408,0.0,7739.273,5856.170
43797,SIN,SISTEMA INTERLIGADO NACIONAL,2025-12-31 23:00:00,54541.992,8267.268,11872.449,1.0,74682.710,0.000
43798,SE,SUDESTE/CENTRO-OESTE,2025-12-31 23:00:00,27730.875,5679.691,187.179,0.0,42418.294,-8820.549
43799,S,SUL,2025-12-31 23:00:00,11883.133,158.417,587.178,1.0,11458.166,1171.562


### Resultado da caracterização inicial

A base de 2025 possui **43.800 registros e 9 variáveis**.

Na leitura inicial:

- as variáveis de identificação e instante foram carregadas como texto;
- as variáveis de geração, carga e intercâmbio foram reconhecidas como numéricas;
- não foram identificados valores nulos pela inspeção inicial com `info()`;
- a variável `din_instante` ainda se encontra no formato textual e será convertida para o tipo `datetime` na etapa de preparação dos dados.

A quantidade de registros observada é compatível com uma série horária anual contendo cinco categorias de sistema/subsistema. Essa hipótese será validada programaticamente no diagnóstico de cobertura temporal e granularidade.

## 6. Diagnóstico da qualidade dos dados

Antes de realizar qualquer transformação, será avaliada a qualidade da base original.

O diagnóstico será estruturado em cinco dimensões:

1. **Completude:** verifica a existência de valores ausentes;
2. **Unicidade:** verifica a existência de registros duplicados;
3. **Cobertura:** verifica a continuidade e a abrangência temporal dos dados;
4. **Validade:** verifica se os valores respeitam as regras esperadas para cada variável;
5. **Plausibilidade:** investiga valores extremos ou comportamentos que possam exigir análise adicional.

Nenhum registro será removido ou alterado durante esta etapa. O objetivo é identificar e documentar eventuais problemas antes de definir os tratamentos necessários.

### 6.1 Granularidade e categorias da base

A granularidade esperada é de **um registro por sistema/subsistema em cada instante horário**.

Inicialmente, serão verificadas as categorias existentes nas variáveis `id_subsistema` e `nom_subsistema`, bem como a quantidade de registros associada a cada uma.

In [53]:
# Verifica as combinações existentes entre o identificador e o nome de cada sistema/subsistema
categorias_subsistemas = (
    df_2025[["id_subsistema", "nom_subsistema"]]
    .drop_duplicates()
    .sort_values("id_subsistema")
    .reset_index(drop=True)
)

categorias_subsistemas

,id_subsistema,nom_subsistema
0,N,NORTE
1,NE,NORDESTE
2,S,SUL
3,SE,SUDESTE/CENTRO-OESTE
4,SIN,SISTEMA INTERLIGADO NACIONAL


In [54]:
# Conta a quantidade de registros existente para cada sistema/subsistema
quantidade_por_subsistema = (
    df_2025
    .groupby(["id_subsistema", "nom_subsistema"])
    .size()
    .reset_index(name="quantidade_registros")
)

quantidade_por_subsistema

,id_subsistema,nom_subsistema,quantidade_registros
0,N,NORTE,8760
1,NE,NORDESTE,8760
2,S,SUL,8760
3,SE,SUDESTE/CENTRO-OESTE,8760
4,SIN,SISTEMA INTERLIGADO NACIONAL,8760


### 6.2 Completude

A completude avalia a presença de dados ausentes nas variáveis da base.

Para cada variável, serão calculados o número absoluto e o percentual de valores ausentes em relação ao total de registros.


In [55]:
# Calcula a quantidade de valores ausentes em cada variável
quantidade_nulos = df_2025.isna().sum()

# Calcula o percentual de valores ausentes
percentual_nulos = df_2025.isna().mean() * 100

# Consolida os resultados em uma tabela
diagnostico_nulos = pd.DataFrame({
    "quantidade_nulos": quantidade_nulos,
    "percentual_nulos": percentual_nulos
})

diagnostico_nulos

,quantidade_nulos,percentual_nulos
id_subsistema,0,0.0
nom_subsistema,0,0.0
din_instante,0,0.0
val_gerhidraulica,0,0.0
val_gertermica,0,0.0
val_gereolica,0,0.0
val_gersolar,0,0.0
val_carga,0,0.0
val_intercambio,0,0.0


### 6.3 Unicidade

Considerando a granularidade esperada da base, cada sistema/subsistema deve possuir apenas um registro para cada instante horário.

Assim, a combinação

$$
(\text{id\_subsistema},\ \text{din\_instante})
$$

será considerada uma **chave candidata** da base.

Será verificada a existência de registros repetidos para essa combinação.

In [56]:
# Define as colunas que formam a chave candidata da base
chave = ["id_subsistema", "din_instante"]

# Identifica todos os registros envolvidos em duplicidades da chave
registros_duplicados = df_2025[
    df_2025.duplicated(subset=chave, keep=False)
].sort_values(chave)

print(f"Registros envolvidos em duplicidades: {len(registros_duplicados):,}")

Registros envolvidos em duplicidades: 0


### 6.4 Preparação da variável temporal para o diagnóstico de cobertura

A variável `din_instante` foi inicialmente carregada como texto.

Para avaliar adequadamente a cobertura e a continuidade da série temporal, será criada uma cópia da base e a variável será convertida para o tipo `datetime`.

A base carregada originalmente será preservada nesta etapa.

In [57]:
# Cria uma cópia da base para as etapas de diagnóstico e preparação
df = df_2025.copy()

# Converte o instante de texto para datetime.
# errors="coerce" transforma em NaT eventuais valores que não possam ser interpretados como data/hora.
df["din_instante"] = pd.to_datetime(
    df["din_instante"],
    errors="coerce"
)

# Verifica o resultado da conversão
print(f"Tipo da variável: {df['din_instante'].dtype}")
print(f"Datas inválidas após conversão: {df['din_instante'].isna().sum():,}")

Tipo da variável: datetime64[us]
Datas inválidas após conversão: 0


### 6.5 Cobertura temporal

A cobertura temporal verifica se os dados abrangem o período esperado e se cada sistema/subsistema possui uma série horária completa.

Para cada categoria serão avaliados:

- primeiro instante disponível;
- último instante disponível;
- quantidade total de registros;
- quantidade de instantes distintos.

In [58]:
# Resume a cobertura temporal de cada sistema/subsistema
cobertura_temporal = (
    df
    .groupby(["id_subsistema", "nom_subsistema"])
    .agg(
        primeiro_instante=("din_instante", "min"),
        ultimo_instante=("din_instante", "max"),
        quantidade_registros=("din_instante", "size"),
        instantes_distintos=("din_instante", "nunique")
    )
    .reset_index()
)

cobertura_temporal

,id_subsistema,nom_subsistema,primeiro_instante,ultimo_instante,quantidade_registros,instantes_distintos
0,N,NORTE,2025-01-01,2025-12-31 23:00:00,8760,8760
1,NE,NORDESTE,2025-01-01,2025-12-31 23:00:00,8760,8760
2,S,SUL,2025-01-01,2025-12-31 23:00:00,8760,8760
3,SE,SUDESTE/CENTRO-OESTE,2025-01-01,2025-12-31 23:00:00,8760,8760
4,SIN,SISTEMA INTERLIGADO NACIONAL,2025-01-01,2025-12-31 23:00:00,8760,8760


### Resultado parcial do diagnóstico de qualidade

Até esta etapa, a base apresentou estrutura consistente com a granularidade esperada.

Foram identificadas cinco categorias de sistema/subsistema:

- Norte (`N`);
- Nordeste (`NE`);
- Sul (`S`);
- Sudeste/Centro-Oeste (`SE`);
- Sistema Interligado Nacional (`SIN`).

Cada categoria possui **8.760 registros**, totalizando **43.800 observações** na base.

Também foram verificados:

- **0 valores ausentes** nas variáveis analisadas;
- **0 registros duplicados** para a chave candidata $(id\_subsistema,\ din\_instante)$;
- **0 valores inválidos** na conversão da variável temporal;
- período compreendido entre **01/01/2025 00:00** e **31/12/2025 23:00**;
- **8.760 instantes distintos** para cada sistema/subsistema.

Esses resultados indicam elevada consistência estrutural da base. Entretanto, antes de concluir o diagnóstico de qualidade, ainda serão verificadas a continuidade da frequência horária, a validade dos valores numéricos e a presença de valores extremos ou comportamentos que necessitem investigação.

### 6.6 Continuidade da série temporal

Embora cada sistema/subsistema apresente 8.760 instantes distintos entre o início e o fim de 2025, será verificada explicitamente a continuidade da série temporal.

Para isso, será criada uma sequência contendo todas as horas esperadas entre:

$$
01/01/2025\ 00{:}00
$$

e

$$
31/12/2025\ 23{:}00
$$

A sequência esperada será comparada aos instantes observados para cada sistema/subsistema.

In [59]:
# Cria a sequência completa de horas esperadas para o ano de 2025
horas_esperadas = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-12-31 23:00:00",
    freq="h"
)

print(f"Quantidade de horas esperadas: {len(horas_esperadas):,}")

Quantidade de horas esperadas: 8,760


In [60]:
# Verifica, para cada sistema/subsistema, se existem horas ausentes
resultado_cobertura = []

for id_subsistema, grupo in df.groupby("id_subsistema"):

    horas_observadas = pd.DatetimeIndex(grupo["din_instante"].unique())

    horas_ausentes = horas_esperadas.difference(horas_observadas)

    resultado_cobertura.append({
        "id_subsistema": id_subsistema,
        "horas_esperadas": len(horas_esperadas),
        "horas_observadas": len(horas_observadas),
        "horas_ausentes": len(horas_ausentes)
    })

diagnostico_cobertura = pd.DataFrame(resultado_cobertura)

diagnostico_cobertura

,id_subsistema,horas_esperadas,horas_observadas,horas_ausentes
0,N,8760,8760,0
1,NE,8760,8760,0
2,S,8760,8760,0
3,SE,8760,8760,0
4,SIN,8760,8760,0


### 6.7 Validade dos valores numéricos

Nesta etapa serão avaliados os valores das variáveis numéricas de acordo com sua natureza.

Para as variáveis de geração e carga, serão investigados valores negativos, nulos e iguais a zero.

Para o intercâmbio, valores positivos e negativos são permitidos e possuem significado associado ao sentido do fluxo de energia. Portanto, o sinal do intercâmbio não será considerado uma inconsistência.

O objetivo desta etapa é identificar valores que necessitem investigação, sem realizar exclusões ou substituições automáticas.

In [61]:
# Variáveis relacionadas à geração
colunas_geracao = [
    "val_gerhidraulica",
    "val_gertermica",
    "val_gereolica",
    "val_gersolar"
]

# Variáveis numéricas utilizadas na análise
colunas_numericas = colunas_geracao + [
    "val_carga",
    "val_intercambio"
]

In [62]:
# Resume mínimo, máximo e quantidade de valores negativos, iguais a zero e positivos
diagnostico_validade = []

for coluna in colunas_numericas:

    diagnostico_validade.append({
        "variavel": coluna,
        "minimo": df[coluna].min(),
        "maximo": df[coluna].max(),
        "negativos": (df[coluna] < 0).sum(),
        "zeros": (df[coluna] == 0).sum(),
        "positivos": (df[coluna] > 0).sum()
    })

diagnostico_validade = pd.DataFrame(diagnostico_validade)

diagnostico_validade

,variavel,minimo,maximo,negativos,zeros,positivos
0,val_gerhidraulica,814.225,85895.758,0,0,43800
1,val_gertermica,19.000,16795.298,0,0,43800
2,val_gereolica,0.000,25064.762,0,77,43723
3,val_gersolar,0.000,42551.688,0,9828,33972
4,val_carga,6015.876,106148.660,0,0,43800
5,val_intercambio,-13840.637,13591.512,19692,5328,18780


### 6.8 Plausibilidade e estatísticas descritivas

A análise de plausibilidade busca identificar valores extremos ou comportamentos que mereçam investigação.

Valores elevados ou baixos não serão classificados automaticamente como erros ou `outliers`, pois podem representar condições reais de operação do sistema.

Serão inicialmente avaliadas medidas de posição e dispersão das variáveis numéricas.

In [63]:
# Calcula estatísticas descritivas das variáveis numéricas
estatisticas_descritivas = (
    df[colunas_numericas]
    .describe()
    .T
)

estatisticas_descritivas

,count,mean,std,min,25%,50%,75%,max
val_gerhidraulica,43800.0,18440.109956,17904.113312,814.225,3864.64125,11405.3705,29657.60650,85895.758
val_gertermica,43800.0,4081.346578,4018.260360,19.000,981.22100,2455.1585,6562.03400,16795.298
val_gereolica,43800.0,5267.644285,6957.615686,0.000,153.85225,604.6595,10501.16225,25064.762
val_gersolar,43800.0,4193.271983,7510.634510,0.000,0.27700,215.4890,4958.79550,42551.688
val_carga,43800.0,31843.562801,27591.836714,6015.876,11083.30475,14760.8160,48845.60075,106148.660
val_intercambio,43800.0,138.810115,5250.268230,-13840.637,-3320.14650,0.0000,2668.82625,13591.512


### 6.9 Estatísticas descritivas por sistema/subsistema

As estatísticas descritivas anteriores consideram simultaneamente os quatro subsistemas e o agregado do SIN.

Como as magnitudes dessas séries são diferentes, essa avaliação pode produzir medidas de dispersão pouco representativas.

Por esse motivo, as estatísticas serão também analisadas separadamente para cada sistema/subsistema. Essa abordagem permite avaliar a plausibilidade dos valores respeitando as diferentes escalas das séries.

In [64]:
# Calcula as principais estatísticas descritivas das variáveis numéricas separadamente para cada sistema/subsistema
estatisticas_por_subsistema = (
    df
    .groupby(["id_subsistema", "nom_subsistema"])[colunas_numericas]
    .agg(["min", "mean", "median", "max"])
)

estatisticas_por_subsistema

val_gerhidraulica                \
                                                         min          mean   
id_subsistema nom_subsistema                                                 
N             NORTE                                 1837.833   7578.510361   
NE            NORDESTE                              1750.448   3228.561963   
S             SUL                                    814.225   8293.455796   
SE            SUDESTE/CENTRO-OESTE                 12361.219  26999.746769   
SIN           SISTEMA INTERLIGADO NACIONAL         20915.441  46100.274889   

                                                                   \
                                                median        max   
id_subsistema nom_subsistema                                        
N             NORTE                          5930.0080  19474.506   
NE            NORDESTE                       2600.2855   7395.399   
S             SUL                            8159.7675  15696.087   
SE            SUDESTE/CENTRO-OESTE          26387.1495  48893.056   
SIN           SISTEMA INTERLIGADO NACIONAL  44468.8620  85895.758   

                                           val_gertermica                \
                                                      min          mean   
id_subsistema nom_subsistema                                              
N             NORTE                               155.507   2106.748315   
NE            NORDESTE                            140.413    771.209755   
S             SUL                                  19.000   1065.209762   
SE            SUDESTE/CENTRO-OESTE               3118.961   6260.198613   
SIN           SISTEMA INTERLIGADO NACIONAL       4863.171  10203.366445   

                                                                   \
                                                median        max   
id_subsistema nom_subsistema                                        
N             NORTE                          2205.6800   3486.370   
NE            NORDESTE                        398.1105   3309.733   
S             SUL                            1086.7430   2153.590   
SE            SUDESTE/CENTRO-OESTE           5941.4600  11075.457   
SIN           SISTEMA INTERLIGADO NACIONAL  10166.0235  16795.298   

                                           val_gereolica                ...  \
                                                     min          mean  ...   
id_subsistema nom_subsistema                                            ...   
N             NORTE                                0.000    212.354963  ...   
NE            NORDESTE                           569.397  12189.350909  ...   
S             SUL                                 15.047    698.753075  ...   
SE            SUDESTE/CENTRO-OESTE                 1.000     68.651765  ...   
SIN           SISTEMA INTERLIGADO NACIONAL       619.469  13169.110712  ...   

                                           val_gersolar             val_carga  \
                                                 median        max        min   
id_subsistema nom_subsistema                                                    
N             NORTE                             59.6045   2438.820   6015.876   
NE            NORDESTE                         358.8080  13834.613   9257.394   
S             SUL                               83.9575   6950.832   7078.137   
SE            SUDESTE/CENTRO-OESTE             555.9905  21933.872  29591.554   
SIN           SISTEMA INTERLIGADO NACIONAL    1396.7435  42551.688  55129.830   

                                                                      \
                                                    mean      median   
id_subsistema nom_subsistema                                           
N             NORTE                          8311.701382   8282.7690   
NE            NORDESTE                      13266.864794  13338.0870   
S             SUL                           13804.431484  13906.5960 

### 6.10 Consistência entre o SIN e os subsistemas

A base contém simultaneamente registros dos quatro subsistemas e do Sistema Interligado Nacional (SIN).

Como verificação adicional de consistência, será comparado o valor informado para o SIN com a soma dos valores dos subsistemas Norte, Nordeste, Sul e Sudeste/Centro-Oeste em cada instante.

Para uma variável $X$, será calculada a diferença:

$$
\varepsilon_{X,t}
=
X_{SIN,t}
-
\sum_{s \in \{N,NE,S,SE\}} X_{s,t}
$$

Essa comparação será realizada para carga e para as diferentes fontes de geração.

Pequenas diferenças podem ocorrer em função de critérios de consolidação ou precisão numérica. Diferenças relevantes, entretanto, deverão ser investigadas antes da continuidade da análise.

In [65]:
# Variáveis que serão utilizadas para verificar a consistência
# entre o valor informado para o SIN e a soma dos subsistemas
variaveis_consistencia = [
    "val_gerhidraulica",
    "val_gertermica",
    "val_gereolica",
    "val_gersolar",
    "val_carga"
]

In [66]:
# Separa os registros correspondentes ao SIN
df_sin = (
    df[df["id_subsistema"] == "SIN"]
    .set_index("din_instante")
    .sort_index()
)

# Seleciona apenas os quatro subsistemas
df_subsistemas = df[
    df["id_subsistema"].isin(["N", "NE", "S", "SE"])
]

In [67]:
# Soma, para cada instante, os valores observados nos quatro subsistemas
soma_subsistemas = (
    df_subsistemas
    .groupby("din_instante")[variaveis_consistencia]
    .sum()
    .sort_index()
)

In [68]:
# Calcula a diferença entre o valor informado para o SIN e a soma dos quatro subsistemas
diferencas_sin = (
    df_sin[variaveis_consistencia]
    - soma_subsistemas[variaveis_consistencia]
)

# Resume as diferenças observadas
diagnostico_consistencia = pd.DataFrame({
    "diferenca_minima": diferencas_sin.min(),
    "diferenca_media": diferencas_sin.mean(),
    "diferenca_maxima": diferencas_sin.max(),
    "maior_diferenca_absoluta": diferencas_sin.abs().max()
})

diagnostico_consistencia

,diferenca_minima,diferenca_media,diferenca_maxima,maior_diferenca_absoluta
val_gerhidraulica,-1.455192e-11,1.120777e-08,4.001049e-08,4.001049e-08
val_gertermica,-3.637979e-12,8.012572e-09,4.000140e-08,4.000140e-08
val_gereolica,-3.637979e-12,6.067341e-09,3.000241e-08,3.000241e-08
val_gersolar,-7.275958e-12,6.039975e-09,4.000321e-08,4.000321e-08
val_carga,-4.999970e-03,4.611422e-04,5.000040e-03,5.000040e-03


### Resultado da verificação de consistência do agregado SIN

A comparação entre os valores informados para o SIN e a soma dos quatro subsistemas apresentou diferenças desprezíveis.

Para as diferentes fontes de geração, as maiores diferenças absolutas observadas ficaram na ordem de $10^{-8}$ MWmed.

Para a carga, a maior diferença absoluta foi de aproximadamente:

$$
0{,}005\ \text{MWmed}
$$

valor desprezível em relação à magnitude da carga do sistema.

Assim, os resultados indicam elevada consistência entre os registros individuais dos subsistemas e os valores consolidados disponibilizados para o SIN. As pequenas diferenças observadas são compatíveis com efeitos de precisão numérica e arredondamento.

### 6.11 Comportamento do intercâmbio por sistema/subsistema

O intercâmbio possui natureza distinta das variáveis de geração e carga.

Valores positivos, negativos e iguais a zero podem possuir significado operacional e não devem ser classificados como inconsistências apenas com base no sinal.

Por esse motivo, será avaliada inicialmente a distribuição dos sinais e das magnitudes do intercâmbio para cada sistema/subsistema.


In [69]:
# Avalia o comportamento do intercâmbio separadamente
# para cada sistema/subsistema
diagnostico_intercambio = (
    df
    .groupby(["id_subsistema", "nom_subsistema"])
    .agg(
        minimo=("val_intercambio", "min"),
        media=("val_intercambio", "mean"),
        mediana=("val_intercambio", "median"),
        maximo=("val_intercambio", "max"),
        negativos=("val_intercambio", lambda x: (x < 0).sum()),
        zeros=("val_intercambio", lambda x: (x == 0).sum()),
        positivos=("val_intercambio", lambda x: (x > 0).sum())
    )
    .reset_index()
)

diagnostico_intercambio

,id_subsistema,nom_subsistema,minimo,media,mediana,maximo,negativos,zeros,positivos
0,N,NORTE,-4535.008,2169.389135,755.5395,13252.070,3928,0,4832
1,NE,NORDESTE,-7872.617,6263.500632,6844.2185,13591.512,620,0,8140
2,S,SUL,-9760.334,-2445.988169,-2403.4850,6887.991,6443,0,2317
3,SE,SUDESTE/CENTRO-OESTE,-13840.637,-5639.876367,-5781.5385,3300.119,8438,0,322
4,SIN,SISTEMA INTERLIGADO NACIONAL,-1546.200,347.025345,0.0000,2620.330,263,5328,3169


### 6.12 Consistência do balanço energético por sistema/subsistema

Como validação adicional da consistência dos dados, será verificado o fechamento do balanço energético em cada instante.

A geração total será definida como:

$$
G_{total}
=
G_{hidráulica}
+
G_{térmica}
+
G_{eólica}
+
G_{solar}
$$

O atendimento à carga de cada subsistema também depende do intercâmbio líquido de energia.

Como o sinal da variável `val_intercambio` possui significado associado ao sentido do fluxo, inicialmente serão avaliadas as duas convenções possíveis:

$$
\varepsilon_1 =
G_{total} + I - C
$$

e

$$
\varepsilon_2 =
G_{total} - I - C
$$

A convenção que produzir resíduos próximos de zero será considerada compatível com a definição utilizada na base.

Após a identificação dessa convenção, será avaliado o fechamento do balanço para cada sistema/subsistema ao longo de 2025.

In [70]:
# Cria uma cópia para a validação do balanço energético
df_balanco = df.copy()

# Calcula a geração total em cada registro
df_balanco["geracao_total"] = (
    df_balanco["val_gerhidraulica"]
    + df_balanco["val_gertermica"]
    + df_balanco["val_gereolica"]
    + df_balanco["val_gersolar"]
)

# Testa as duas possíveis convenções de sinal do intercâmbio
df_balanco["residuo_convencao_1"] = (
    df_balanco["geracao_total"]
    + df_balanco["val_intercambio"]
    - df_balanco["val_carga"]
)

df_balanco["residuo_convencao_2"] = (
    df_balanco["geracao_total"]
    - df_balanco["val_intercambio"]
    - df_balanco["val_carga"]
)

# Compara a magnitude dos resíduos das duas convenções
comparacao_convencoes = pd.DataFrame({
    "residuo_medio_absoluto": [
        df_balanco["residuo_convencao_1"].abs().mean(),
        df_balanco["residuo_convencao_2"].abs().mean()
    ],
    "residuo_maximo_absoluto": [
        df_balanco["residuo_convencao_1"].abs().max(),
        df_balanco["residuo_convencao_2"].abs().max()
    ]
}, index=["geracao + intercambio - carga",
          "geracao - intercambio - carga"])

comparacao_convencoes


,residuo_medio_absoluto,residuo_maximo_absoluto
geracao + intercambio - carga,7910.396228,27681.274
geracao - intercambio - carga,0.000571,0.010


#### Resultado da identificação da convenção de sinal

A comparação entre as duas convenções possíveis mostrou que o balanço compatível com os dados é:

$$
G_{total} - I - C \approx 0
$$

ou, equivalentemente:

$$
C \approx G_{total} - I
$$

Dessa forma, para a convenção utilizada nesta base, valores positivos de `val_intercambio` representam fluxo líquido de energia saindo do sistema/subsistema, enquanto valores negativos representam fluxo líquido contribuindo para o atendimento da carga local.

In [71]:
# Calcula o resíduo do balanço energético utilizando
# a convenção de sinal identificada nos dados
df_balanco["residuo_balanco"] = (
    df_balanco["geracao_total"]
    - df_balanco["val_intercambio"]
    - df_balanco["val_carga"]
)

# Resume o fechamento do balanço energético
# separadamente para cada sistema/subsistema
diagnostico_balanco = (
    df_balanco
    .groupby(["id_subsistema", "nom_subsistema"])
    .agg(
        residuo_minimo=("residuo_balanco", "min"),
        residuo_medio=("residuo_balanco", "mean"),
        residuo_mediano=("residuo_balanco", "median"),
        residuo_maximo=("residuo_balanco", "max"),
        residuo_medio_absoluto=("residuo_balanco", lambda x: x.abs().mean()),
        maior_residuo_absoluto=("residuo_balanco", lambda x: x.abs().max())
    )
    .reset_index()
)

diagnostico_balanco

,id_subsistema,nom_subsistema,residuo_minimo,residuo_medio,residuo_mediano,residuo_maximo,residuo_medio_absoluto,maior_residuo_absoluto
0,N,NORTE,-4.000140e-08,-3.110725e-09,0.000000e+00,2.000161e-08,6.293591e-09,4.000140e-08
1,NE,NORDESTE,-3.000241e-08,-4.972553e-09,-1.818989e-12,2.000161e-08,6.653418e-09,3.000241e-08
2,S,SUL,-4.000140e-08,-4.917830e-09,-1.818989e-12,2.000161e-08,6.118999e-09,4.000140e-08
3,SE,SUDESTE/CENTRO-OESTE,-4.001049e-08,-7.883429e-09,-9.997166e-09,1.001172e-08,8.247213e-09,4.001049e-08
4,SIN,SISTEMA INTERLIGADO NACIONAL,-1.000000e-02,-5.749429e-04,-1.000000e-03,9.000000e-03,2.855879e-03,1.000000e-02


## 6.13 Conclusão do diagnóstico de qualidade dos dados

O diagnóstico realizado indica que a base de dados de 2025 apresenta **elevada consistência**, não tendo sido identificadas inconsistências que justifiquem a exclusão, imputação ou correção de registros.

### Estrutura e granularidade

A base contém **43.800 registros**, correspondentes a cinco categorias:

- Norte (`N`);
- Nordeste (`NE`);
- Sul (`S`);
- Sudeste/Centro-Oeste (`SE`);
- Sistema Interligado Nacional (`SIN`).

Cada categoria apresenta **8.760 registros**, compatíveis com uma observação para cada hora do ano de 2025.

A combinação

$$
(\text{id\_subsistema},\ \text{din\_instante})
$$

foi validada como chave candidata da base, não tendo sido encontrados registros duplicados.

### Completude e cobertura temporal

Não foram identificados valores ausentes em nenhuma das nove variáveis da base.

A variável temporal foi convertida para o formato `datetime` sem ocorrência de valores inválidos.

Para todos os sistemas/subsistemas, foram observadas as **8.760 horas esperadas**, compreendidas entre:

$$
01/01/2025\ 00{:}00
$$

e

$$
31/12/2025\ 23{:}00
$$

sem identificação de lacunas na série temporal.

### Validade das variáveis numéricas

Não foram observados valores negativos para as variáveis de geração ou carga.

Foram encontrados valores iguais a zero nas séries de geração eólica e solar, considerados compatíveis com a natureza dessas fontes e, portanto, mantidos na base.

A variável de intercâmbio apresentou valores positivos, negativos e iguais a zero, comportamento compatível com sua natureza de fluxo líquido de energia.

### Consistência entre SIN e subsistemas

A comparação entre os valores informados para o SIN e a soma dos quatro subsistemas apresentou diferenças desprezíveis.


### Fechamento do balanço energético

A avaliação do balanço permitiu identificar a seguinte convenção para o intercâmbio:

$$
C \approx G_{total} - I
$$

onde:

$$
G_{total}
=
G_{hidráulica}
+
G_{térmica}
+
G_{eólica}
+
G_{solar}
$$


As diferenças foram desprezíveis frente à magnitude das variáveis analisadas e são compatíveis com efeitos de arredondamento e precisão numérica.

### Decisão sobre o tratamento da base

Com base nos testes realizados, **não foram identificados problemas de qualidade que demandassem remoção, imputação ou correção dos valores originais**.

## 7. Preparação dos dados para análise

Após o diagnóstico de qualidade, não foram identificadas inconsistências que justificassem exclusão, imputação ou correção dos registros originais.

Nesta etapa, a base será preparada para as análises posteriores por meio da criação de variáveis derivadas relacionadas ao tempo e ao balanço energético.

Os principais objetivos desta etapa são:

- preservar os dados originais;
- criar variáveis temporais para facilitar agrupamentos e comparações;
- calcular a geração total;
- calcular a carga líquida como variável complementar;
- calcular a participação de cada fonte na geração total;
- gerar uma base processada e reproduzível para utilização no notebook de análise.

A base processada manterá os registros dos quatro subsistemas e do SIN.

In [72]:
# Cria uma nova base de trabalho a partir do DataFrame que já possui a variável temporal convertida para datetime
df_processado = df.copy()

# Ordena os registros por instante e sistema/subsistema
df_processado = (
    df_processado
    .sort_values(["din_instante", "id_subsistema"])
    .reset_index(drop=True)
)

df_processado.head()

,id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
0,N,NORTE,2025-01-01,6818.532,1576.087,401.685,0.0,7097.751,1698.553
1,NE,NORDESTE,2025-01-01,2219.280,415.154,16070.653,0.0,12251.182,6453.905
2,S,SUL,2025-01-01,7938.930,333.262,1185.180,1.0,10554.603,-1096.231
3,SE,SUDESTE/CENTRO-OESTE,2025-01-01,23801.970,4510.835,5.001,0.0,35374.033,-7056.227
4,SIN,SISTEMA INTERLIGADO NACIONAL,2025-01-01,40778.712,6835.338,17662.519,1.0,65277.570,0.000


### 7.1 Criação das variáveis temporais

A variável `din_instante` contém simultaneamente informações de data e hora.

Para facilitar os agrupamentos e as análises temporais, serão criadas variáveis derivadas contendo:

- data;
- ano;
- mês;
- hora;
- dia da semana;
- indicador de final de semana.

Essas variáveis permitirão avaliar posteriormente padrões horários, mensais e diferenças entre dias úteis e finais de semana.

In [73]:
# Cria variáveis derivadas a partir do instante temporal
df_processado["data"] = df_processado["din_instante"].dt.normalize()
df_processado["ano"] = df_processado["din_instante"].dt.year
df_processado["mes"] = df_processado["din_instante"].dt.month
df_processado["hora"] = df_processado["din_instante"].dt.hour
df_processado["dia_semana_num"] = df_processado["din_instante"].dt.dayofweek

# Cria um indicador de final de semana.
# No pandas, segunda-feira = 0 e domingo = 6.
df_processado["fim_semana"] = df_processado["dia_semana_num"] >= 5

C:\Users\thais.oliveira\AppData\Local\Temp\ipykernel_7684\1118176090.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_processado["dia_semana_num"] = df_processado["din_instante"].dt.dayofweek


In [74]:
# Mapeia o número do dia da semana para seu nome em português
nomes_dias = {
    0: "Segunda-feira",
    1: "Terça-feira",
    2: "Quarta-feira",
    3: "Quinta-feira",
    4: "Sexta-feira",
    5: "Sábado",
    6: "Domingo"
}

df_processado["dia_semana"] = (
    df_processado["dia_semana_num"]
    .map(nomes_dias)
)

### 7.2 Validação das variáveis temporais

Após a criação das variáveis temporais, serão verificadas suas categorias e faixas de valores para confirmar se a transformação ocorreu conforme esperado.

In [75]:
# Resume as faixas das novas variáveis temporais
print(f"Ano: {df_processado['ano'].min()} a {df_processado['ano'].max()}")
print(f"Mês: {df_processado['mes'].min()} a {df_processado['mes'].max()}")
print(f"Hora: {df_processado['hora'].min()} a {df_processado['hora'].max()}")

print("\nDias da semana:")
print(
    df_processado[
        ["dia_semana_num", "dia_semana"]
    ]
    .drop_duplicates()
    .sort_values("dia_semana_num")
    .to_string(index=False)
)

Ano: 2025 a 2025
Mês: 1 a 12
Hora: 0 a 23

Dias da semana:
 dia_semana_num    dia_semana
              0 Segunda-feira
              1   Terça-feira
              2  Quarta-feira
              3  Quinta-feira
              4   Sexta-feira
              5        Sábado
              6       Domingo


### 7.3 Cálculo da geração total

A geração total será calculada como a soma das quatro fontes de geração disponíveis na base:

$$
G_{total}
=
G_{hidráulica}
+
G_{térmica}
+
G_{eólica}
+
G_{solar}
$$

Essa variável será utilizada posteriormente tanto na análise da composição da geração quanto na construção de indicadores derivados.

In [76]:
# Calcula a geração total a partir das quatro fontes
df_processado["geracao_total"] = (
    df_processado["val_gerhidraulica"]
    + df_processado["val_gertermica"]
    + df_processado["val_gereolica"]
    + df_processado["val_gersolar"]
)

### 7.4 Cálculo da carga líquida

A análise principal utilizará a **carga bruta** para identificação dos horários de pico e vale.

Como variável complementar, será calculada também a carga líquida, definida neste projeto como a carga bruta descontada das gerações eólica e solar:

$$
C_{líquida}
=
C_{bruta}
-
G_{eólica}
-
G_{solar}
$$

Essa variável representa uma aproximação da parcela da carga que permanece após descontada a geração das fontes renováveis variáveis consideradas na base.

A carga líquida não será utilizada inicialmente para definir os horários de pico e vale, mas permitirá investigar posteriormente se o perfil de demanda líquida apresenta comportamento temporal diferente da carga bruta.

In [77]:
# Calcula a carga líquida utilizada na análise complementar
df_processado["carga_liquida"] = (
    df_processado["val_carga"]
    - df_processado["val_gereolica"]
    - df_processado["val_gersolar"]
)

### 7.5 Participação das fontes na geração total

Para comparar a composição da geração independentemente da magnitude total do sistema, será calculada a participação percentual de cada fonte na geração total:

$$
P_f
=
\frac{G_f}{G_{total}}
\times 100
$$

onde $G_f$ representa a geração da fonte analisada.

Serão calculadas as participações das fontes:

- hidráulica;
- térmica;
- eólica;
- solar.

In [78]:
# Calcula a participação percentual de cada fonte em relação à geração total
df_processado["part_hidraulica"] = (
    df_processado["val_gerhidraulica"]
    / df_processado["geracao_total"]
    * 100
)

df_processado["part_termica"] = (
    df_processado["val_gertermica"]
    / df_processado["geracao_total"]
    * 100
)

df_processado["part_eolica"] = (
    df_processado["val_gereolica"]
    / df_processado["geracao_total"]
    * 100
)

df_processado["part_solar"] = (
    df_processado["val_gersolar"]
    / df_processado["geracao_total"]
    * 100
)

### 7.6 Validação das variáveis energéticas derivadas

Após o cálculo das novas variáveis, serão realizadas verificações para confirmar sua consistência.

Serão avaliados:

- valores ausentes;
- valores infinitos;
- faixa da geração total;
- faixa da carga líquida;
- soma das participações percentuais das fontes.

Como as quatro participações utilizam a mesma geração total como denominador, espera-se que sua soma seja aproximadamente igual a:

$$
100\%
$$

Pequenas diferenças podem ocorrer em função da precisão numérica.

In [79]:
# Verifica se existe geração total igual ou inferior a zero
quantidade_geracao_nao_positiva = (
    df_processado["geracao_total"] <= 0
).sum()

print(
    f"Registros com geração total <= 0: "
    f"{quantidade_geracao_nao_positiva:,}"
)

Registros com geração total <= 0: 0


In [80]:
# Calcula a soma das participações das quatro fontes
df_processado["soma_participacoes"] = (
    df_processado["part_hidraulica"]
    + df_processado["part_termica"]
    + df_processado["part_eolica"]
    + df_processado["part_solar"]
)

# Avalia a diferença em relação a 100%
erro_participacoes = (
    df_processado["soma_participacoes"] - 100
).abs()

print(
    f"Maior diferença da soma das participações para 100%: "
    f"{erro_participacoes.max():.12f} p.p."
)

Maior diferença da soma das participações para 100%: 0.000000000000 p.p.


In [81]:
# Exibe as faixas das principais variáveis derivadas
resumo_variaveis_derivadas = pd.DataFrame({
    "minimo": df_processado[
        ["geracao_total", "carga_liquida", "soma_participacoes"]
    ].min(),

    "media": df_processado[
        ["geracao_total", "carga_liquida", "soma_participacoes"]
    ].mean(),

    "maximo": df_processado[
        ["geracao_total", "carga_liquida", "soma_participacoes"]
    ].max()
})

resumo_variaveis_derivadas

,minimo,media,maximo
geracao_total,2302.768,31982.372802,106707.645
carga_liquida,-10591.242,22382.646534,92483.358
soma_participacoes,100.000,100.000000,100.000


In [82]:
# Define a ordem das colunas da base processada
colunas_finais = [
    # Identificação
    "id_subsistema",
    "nom_subsistema",

    # Tempo
    "din_instante",
    "data",
    "ano",
    "mes",
    "hora",
    "dia_semana_num",
    "dia_semana",
    "fim_semana",

    # Geração original
    "val_gerhidraulica",
    "val_gertermica",
    "val_gereolica",
    "val_gersolar",

    # Carga e intercâmbio
    "val_carga",
    "val_intercambio",

    # Variáveis derivadas
    "geracao_total",
    "carga_liquida",
    "part_hidraulica",
    "part_termica",
    "part_eolica",
    "part_solar"
]

df_processado = df_processado[colunas_finais]

### 7.7 Exportação da base processada

Após a preparação e validação das variáveis derivadas, a base será armazenada na pasta `dados/processados`.

O arquivo original permanecerá preservado em `dados/brutos`.

In [ ]:
# Garante a existência da pasta destinada aos dados processados
PASTA_DADOS_PROCESSADOS.mkdir(parents=True, exist_ok=True)

# Define o caminho do arquivo processado
ARQUIVO_PROCESSADO = (
    PASTA_DADOS_PROCESSADOS
    / "balanco_energia_2025_processado.csv"
)

# Salva a base processada
df_processado.to_csv(
    ARQUIVO_PROCESSADO,
    index=False,
    sep=";"
)

print(f"Base processada salva em:\n{ARQUIVO_PROCESSADO}")

Base processada salva em:
c:\OneDrive\OneDrive - Operador Nacional do Sistema Eletrico\Área de Trabalho\Curso Analise de Dados\projeto-integrador\dados\processados\balanco_energia_2025_processado_backup.csv


In [84]:
# Realiza uma leitura de teste do arquivo processado
df_teste = pd.read_csv(
    ARQUIVO_PROCESSADO,
    sep=";",
    parse_dates=["din_instante", "data"]
)

print(f"Linhas:  {df_teste.shape,}")
print(f"Colunas: {df_teste.shape[1]}")

df_teste.head()

Linhas:  ((43800, 22),)
Colunas: 22


,id_subsistema,nom_subsistema,din_instante,data,ano,mes,hora,dia_semana_num,dia_semana,fim_semana,...,val_gereolica,val_gersolar,val_carga,val_intercambio,geracao_total,carga_liquida,part_hidraulica,part_termica,part_eolica,part_solar
0,N,NORTE,2025-01-01,2025-01-01,2025,1,0,2,Quarta-feira,False,...,401.685,0.0,7097.751,1698.553,8796.304,6696.066,77.515875,17.917605,4.566520,0.000000
1,NE,NORDESTE,2025-01-01,2025-01-01,2025,1,0,2,Quarta-feira,False,...,16070.653,0.0,12251.182,6453.905,18705.087,-3819.471,11.864580,2.219471,85.915949,0.000000
2,S,SUL,2025-01-01,2025-01-01,2025,1,0,2,Quarta-feira,False,...,1185.180,1.0,10554.603,-1096.231,9458.372,9368.423,83.935481,3.523460,12.530486,0.010573
3,SE,SUDESTE/CENTRO-OESTE,2025-01-01,2025-01-01,2025,1,0,2,Quarta-feira,False,...,5.001,0.0,35374.033,-7056.227,28317.806,35369.032,84.053016,15.929324,0.017660,0.000000
4,SIN,SISTEMA INTERLIGADO NACIONAL,2025-01-01,2025-01-01,2025,1,0,2,Quarta-feira,False,...,17662.519,1.0,65277.570,0.000,65277.569,47614.051,62.469716,10.471190,27.057562,0.001532


## 8. Conclusão da preparação dos dados

A etapa de preparação resultou em uma base estruturada e pronta para utilização nas análises do projeto.

A partir dos dados originais foram criadas variáveis temporais para permitir análises por data, hora, mês e dia da semana, além de variáveis derivadas relacionadas à composição do atendimento à carga.

Foram calculadas:

- geração total;
- carga líquida;
- participação percentual da geração hidráulica;
- participação percentual da geração térmica;
- participação percentual da geração eólica;
- participação percentual da geração solar.

As variáveis derivadas foram validadas, não sendo identificadas inconsistências que exigissem alterações adicionais.

A base processada mantém os registros dos quatro subsistemas e do SIN, totalizando **43.800 registros e 22 variáveis**.

O arquivo resultante foi armazenado em:

`dados/processados/balanco_energia_2025_processado.csv`

A partir deste ponto, a etapa de análise será desenvolvida separadamente no notebook:

`02_analise_pico_vale.ipynb`